In [21]:
import os
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd


class SpotifyAnalyzer:
    def __init__(self, client_id=None, client_secret=None):
        """
        Initialize Spotify API client with client_id and client_secret.
        If not passed, it reads from environment variables.
        """
        client_id = client_id or os.getenv("SPOTIPY_CLIENT_ID")
        client_secret = client_secret or os.getenv("SPOTIPY_CLIENT_SECRET")

        if not client_id or not client_secret:
            raise ValueError("Please provide client_id and client_secret "
                             "or set them as environment variables.")

        auth_manager = SpotifyClientCredentials(
            client_id=client_id,
            client_secret=client_secret
        )
        self.sp = spotipy.Spotify(auth_manager=auth_manager)

    def get_user_playlists(self, user_id, save_csv=True, filename="spotify_playlists.csv"):
        """
        Fetch all playlists of a given user.
        """
        playlists = self.sp.user_playlists(user_id)
        data = []

        while playlists:
            for playlist in playlists['items']:
                data.append({
                    "playlist_name": playlist['name'],
                    "playlist_id": playlist['id'],
                    "playlist_uri": playlist['uri'],
                    "owner": playlist['owner']['display_name'] if playlist['owner'] else None,
                    "tracks_count": playlist['tracks']['total']
                })
            playlists = self.sp.next(playlists) if playlists['next'] else None

        df = pd.DataFrame(data)
        if save_csv and not df.empty:
            df.to_csv(filename, index=False)
            print(f" Saved: {filename}")
        return df

    def search_artist(self, name, save_csv=True):
        """
        Search an artist by name and return details.
        """
        results = self.sp.search(q=f'artist:{name}', type='artist')
        items = results['artists']['items']
        data = []

        for artist in items:
            data.append({
                "artist_name": artist['name'],
                "artist_id": artist['id'],
                "artist_uri": artist['uri'],
                "followers": artist['followers']['total'],
                "genres": ", ".join(artist['genres']),
                "popularity": artist['popularity'],
                "image_url": artist['images'][0]['url'] if artist['images'] else None
            })

        df = pd.DataFrame(data)
        if save_csv and not df.empty:
            filename = f"{name.lower().replace(' ', '_')}_artists.csv"
            df.to_csv(filename, index=False)
            print(f" Saved: {filename}")
        return df

    def get_artist_albums(self, artist_uri, artist_name, save_csv=True):
        """
        Fetch all albums of an artist.
        """
        results = self.sp.artist_albums(artist_uri, album_type='album')
        albums = results['items']
        while results['next']:
            results = self.sp.next(results)
            albums.extend(results['items'])

        data = []
        for album in albums:
            data.append({
                "artist": artist_name,
                "album_name": album['name'],
                "release_date": album['release_date'],
                "total_tracks": album['total_tracks'],
                "album_url": album['external_urls']['spotify']
            })

        df = pd.DataFrame(data)
        if save_csv and not df.empty:
            filename = f"{artist_name.lower().replace(' ', '_')}_albums.csv"
            df.to_csv(filename, index=False)
            print(f" Saved: {filename}")
        return df

    def get_artist_top_tracks(self, artist_uri, artist_name, save_csv=True):
        """
        Fetch top 10 tracks of an artist.
        """
        results = self.sp.artist_top_tracks(artist_uri)
        data = []
        for track in results['tracks'][:10]:
            data.append({
                "artist": artist_name,
                "track_name": track['name'],
                "popularity": track['popularity'],
                "preview_url": track['preview_url'],
                "cover_art": track['album']['images'][0]['url'] if track['album']['images'] else None,
                "album_name": track['album']['name'],
                "release_date": track['album']['release_date']
            })

        df = pd.DataFrame(data)
        if save_csv and not df.empty:
            filename = f"{artist_name.lower().replace(' ', '_')}_top_tracks.csv"
            df.to_csv(filename, index=False)
            print(f" Saved: {filename}")
        return df


In [22]:
import os

os.environ["SPOTIPY_CLIENT_ID"] = "c97d261239c2476d9b85f08179450fc1"
os.environ["SPOTIPY_CLIENT_SECRET"] = "9262159ff2264fefa2568620aab11d97"


In [23]:
spotify = SpotifyAnalyzer()

In [24]:
df_artist = spotify.search_artist("Arijit Singh")
df_artist

 Saved: arijit_singh_artists.csv


,artist_name,artist_id,artist_uri,followers,genres,popularity,image_url
0,Arijit Singh,4YRxDV8wJFPHPTeXepOstw,spotify:artist:4YRxDV8wJFPHPTeXepOstw,157248833,"hindi pop, bollywood, desi, bangla pop",93,https://i.scdn.co/image/ab6761610000e5eb5ba2d7...
1,Arijit Singh,6zrQkGA8EvD1Y4z35lIc2e,spotify:artist:6zrQkGA8EvD1Y4z35lIc2e,3404,"bangla pop, bhajan, devotional",27,https://i.scdn.co/image/ab67616d0000b273fff068...
2,"Hariharan, Swarnalatha, Kumar Sanu, Sapna Mukh...",2b84vRVbe4QTox0pMqAC6W,spotify:artist:2b84vRVbe4QTox0pMqAC6W,998,,10,https://i.scdn.co/image/ab67616d0000b2737bbcbc...
3,"Vishal-Shekhar,Arijit Singh,Shilpa Rao,Kumaar",0l1GEW74dNahpWUZ6VY2aG,spotify:artist:0l1GEW74dNahpWUZ6VY2aG,38,,0,None
4,Arijit Singh,11P6oIAn8olEk5FJSbhAUy,spotify:artist:11P6oIAn8olEk5FJSbhAUy,20,,0,https://i.scdn.co/image/ab67616d0000b273bf7005...
5,Arijit Singh,1Y7GoYOL4iwZNPULQbKqkF,spotify:artist:1Y7GoYOL4iwZNPULQbKqkF,204,,0,https://i.scdn.co/image/ab67616d0000b2733a28ea...
6,"Sachin-Jigar,Arijit Singh,Neeti Mohan,Vayu",4hMvXl2N76KCluwwzziGpq,spotify:artist:4hMvXl2N76KCluwwzziGpq,14,,0,None
7,Arijitt singh,63tnhrqJroQi6keC7fU1rQ,spotify:artist:63tnhrqJroQi6keC7fU1rQ,30,,0,https://i.scdn.co/image/ab67616d0000b27344ce83...
8,"Vishal-Shekhar,Arijit Singh,Jaideep Sahni",5g7Oo0qcaRJpXaSTVwdRv0,spotify:artist:5g7Oo0qcaRJpXaSTVwdRv0,6,,0,None
9,"Sachin-Jigar,Arijit Singh,Priya Saraiya",69BzYn1D0VM66nWt01CwlQ,spotify:artist:69BzYn1D0VM66nWt01CwlQ,77,,0,None


In [25]:
artist_uri = "spotify:artist:4YRxDV8wJFPHPTeXepOstw"  # Arijit Singh
df_albums = spotify.get_artist_albums(artist_uri, "Arijit Singh")
df_albums

 Saved: arijit_singh_albums.csv


,artist,album_name,release_date,total_tracks,album_url
0,Arijit Singh,Bollywood Reimagined: Bollywood Fusion Remix,2025-05-24,8,https://open.spotify.com/album/470a4sx8xtsGjn4...
1,Arijit Singh,The Arijit Singh Collection Vol.3,2025-02-07,21,https://open.spotify.com/album/63OVAw3XKXePV92...
2,Arijit Singh,Arijit Singh Top Sad Love Hits,2024-04-26,17,https://open.spotify.com/album/0os2iPNHyo2yBFi...
3,Arijit Singh,Arijit Singh - King of Sad Hits,2024-04-26,15,https://open.spotify.com/album/2WcskJsPOs5HuFH...
4,Arijit Singh,Arijit Singh - Sad Love Songs,2024-04-25,11,https://open.spotify.com/album/7nwUTtRTuz7ZvJA...
5,Arijit Singh,Arijit Singh - Reprise and Unplugged Hits,2024-04-25,9,https://open.spotify.com/album/2Aa7I2sffdVuuBt...
6,Arijit Singh,The Arijit Singh Collection Vol.2,2024-04-24,29,https://open.spotify.com/album/0JWpJ4hPD4WX0FO...
7,Arijit Singh,Arijit Singh Heartbreak Hits,2024-04-24,15,https://open.spotify.com/album/6L8x1JeBN5nXqZF...
8,Arijit Singh,Arijit Singh - Unheard Gems,2024-04-24,14,https://open.spotify.com/album/68fMXL8ncgfgzpE...
9,Arijit Singh,Dil Jhoom With Arijit Singh,2024-02-12,15,https://open.spotify.com/album/0RGri1OR1Q1bk4K...


In [26]:
df_top_tracks = spotify.get_artist_top_tracks(artist_uri, "Arijit Singh")
df_top_tracks

 Saved: arijit_singh_top_tracks.csv


,artist,track_name,popularity,preview_url,cover_art,album_name,release_date
0,Arijit Singh,"Dhun (From ""Saiyaara"")",81,None,https://i.scdn.co/image/ab67616d0000b273781faf...,"Dhun (From ""Saiyaara"")",2025-07-01
1,Arijit Singh,Apna Bana Le,75,None,https://i.scdn.co/image/ab67616d0000b273b85b4e...,Bhediya (Original Motion Picture Soundtrack),2022-12-06
2,Arijit Singh,Aavan Jaavan,64,None,https://i.scdn.co/image/ab67616d0000b2737c2203...,WAR 2,2025-08-16
3,Arijit Singh,"Tujhe Kitna Chahne Lage (From ""Kabir Singh"")",78,None,https://i.scdn.co/image/ab67616d0000b273ba03ff...,"Tujhe Kitna Chahne Lage (From ""Kabir Singh"")",2019-05-31
4,Arijit Singh,Tum Hi Ho,77,None,https://i.scdn.co/image/ab67616d0000b273640472...,Aashiqui 2,2013-04-06
5,Arijit Singh,"Humdard (From ""Ek Villain"")",56,None,https://i.scdn.co/image/ab67616d0000b2737569cb...,Love Forever With Arijit Singh,2017-06-07
6,Arijit Singh,"Agar Tum Saath Ho (From ""Tamasha"")",68,None,https://i.scdn.co/image/ab67616d0000b27325f762...,Love Wali Feeling: Valentine Special,2017-02-10
7,Arijit Singh,"Sajni (From ""Laapataa Ladies"")",77,None,https://i.scdn.co/image/ab67616d0000b273d5f437...,"Sajni (From ""Laapataa Ladies"")",2024-02-12
8,Arijit Singh,"Mast Magan (From ""2 States)",43,None,https://i.scdn.co/image/ab67616d0000b2732d2263...,Love Dose Arijit Singh,2016-02-11
9,Arijit Singh,"Satranga (From ""ANIMAL"")",77,None,https://i.scdn.co/image/ab67616d0000b273021d70...,"Satranga (From ""ANIMAL"")",2023-10-27


In [27]:
df_playlists = spotify.get_user_playlists("spotify") 
df_playlists

 Saved: spotify_playlists.csv


,playlist_name,playlist_id,playlist_uri,owner,tracks_count
0,Classic Honky Tonk,0NfjMqrzcGKVsbYZmhf4Md,spotify:playlist:0NfjMqrzcGKVsbYZmhf4Md,Spotify,50
1,Nicholas Sparks | Songs from the Soundtracks,1scnlLVq91NGtsA9sh0hfw,spotify:playlist:1scnlLVq91NGtsA9sh0hfw,Spotify,53
2,1960s Nostalgia,2NFOUmp2wyR5CrXtKDkUkB,spotify:playlist:2NFOUmp2wyR5CrXtKDkUkB,Spotify,67
3,Breakup Blues,1o2bTwofazfzElA5mXGf2t,spotify:playlist:1o2bTwofazfzElA5mXGf2t,Spotify,44
4,I Hate My Job.,6cdV0hVW2suJaMOxzwE46S,spotify:playlist:6cdV0hVW2suJaMOxzwE46S,Spotify,38
...,...,...,...,...,...
341,Most Listened To British Dads on Spotify,1k9jG0FUp7BcrAF1MZSabO,spotify:playlist:1k9jG0FUp7BcrAF1MZSabO,Spotify,20
342,dw-c,5ji4GZJpll6twskFvKxiHx,spotify:playlist:5ji4GZJpll6twskFvKxiHx,Spotify,50
343,dw_g,40VxbK9NqccdUDUpiUXmbp,spotify:playlist:40VxbK9NqccdUDUpiUXmbp,Spotify,30
344,Top Shower Songs,0RTz1jFo5BXGPfI8eVf8sj,spotify:playlist:0RTz1jFo5BXGPfI8eVf8sj,Spotify,100
